# 💉 Build APK — Tính Dịch Truyền (Dr. Nểm CCĐK)

Notebook build APK Android từ app Kivy trên Google Colab — **không cần cài WSL, Docker, hay gì trên máy Windows**.

## 📋 Cách dùng
1. **Upload 2 file** vào Colab:
   - `kivy_app.py` (code app Kivy)
   - `buildozer.spec` (config build)
2. **Chạy tuần tự 6 bước** theo thứ tự, đợi mỗi cell xong mới chạy tiếp.
3. **APK tải về tự động** ở Bước 6.

## ⏱️ Thời gian dự kiến
- Lần đầu: **30-45 phút** (tải SDK + NDK + compile Kivy)
- Lần 2 trở đi: **10-15 phút** (Colab giữ cache)

> 💡 **Mẹo quan trọng:** Mở thêm 1 tab Chrome bất kỳ cùng trình duyệt — Colab sẽ giữ session, không bị disconnect giữa chừng.

> ⚠️ **Đừng đổi runtime** (CPU ↔ GPU) trong khi build đang chạy — sẽ reset session mất hết.

## ⚙️ Bước 1: Cài Python 3.11 + Java 17 + buildozer + Cython 3+

**Quan trọng:** Dùng Python 3.11 (không phải 3.14 mặc định của Colab).

Python 3.14 khi build from source trên NDK r28c clang bị fail với:
- `clang: error: unsupported option '-print-multi-os-directory'`
- `lld not found` → linker errors

Python 3.11 build ổn định hơn nhiều trên Colab.

In [ ]:
import os, subprocess
# Bước 1a: Cài Python 3.11 bằng uv (Colab có sẵn uv)
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11.6'], check=True)
result = subprocess.run(['uv', 'python', 'find', '3.11.6'], capture_output=True, text=True)
py311 = result.stdout.strip()
print(f'Python 3.11 ở: {py311}')
# Symlink /usr/local/bin/python3 → python3.11
if os.path.exists('/usr/local/bin/python3'):
    os.remove('/usr/local/bin/python3')
os.symlink(py311, '/usr/local/bin/python3')
print('Đã symlink python3 →', py311)
# Bước 1b: Cài Java 17 + lib build
subprocess.run(['apt', 'install', '-y', '-qq', 'openjdk-17-jdk-headless', 'libffi-dev', 'libssl-dev', 'autoconf', 'libtool', 'pkg-config', 'ninja-build', 'cmake', 'zip', 'unzip'], check=True)
print('Đã cài Java 17 + lib build')
# Bước 1c: Verify Python và Java
print()
r = subprocess.run(['python3', '--version'], capture_output=True, text=True)
print('python3:', r.stdout.strip())
r = subprocess.run('java -version 2>&1', shell=True, capture_output=True, text=True)
print('Java:', (r.stdout.splitlines()[0] if r.stdout else 'unknown'))
# Bước 1d: Tạo venv với Python 3.11
subprocess.run(['uv', 'venv', '/content/venv', '--python', '3.11.6'], check=True)
print('Đã tạo venv tại /content/venv')
# Cài pip vào venv TRƯỚC (uv tạo venv không tự cài pip)
subprocess.run(['uv', 'pip', 'install', '--python', '/content/venv/bin/python', 'pip', 'setuptools', 'wheel'], check=True)
print('Đã cài pip + setuptools + wheel vào venv')
# Cài buildozer + cython vào venv
subprocess.run(['uv', 'pip', 'install', '--python', '/content/venv/bin/python', 'buildozer==1.5.0', 'cython>=3.0'], check=True)
print('Đã cài buildozer + cython vào venv')
# Symlink buildozer CLI để gọi ngắn gọn
if os.path.exists('/usr/local/bin/buildozer'):
    os.remove('/usr/local/bin/buildozer')
os.symlink('/content/venv/bin/buildozer', '/usr/local/bin/buildozer')
print('Đã symlink buildozer')
# Verify buildozer
r = subprocess.run(['buildozer', '--version'], capture_output=True, text=True)
print('Buildozer:', r.stdout.strip())
print()
print('✅ Bước 1 xong — môi trường sẵn sàng (Python 3.11 + Java 17 + Cython 3+)')


## 📂 Bước 2: Upload 2 file `kivy_app.py` và `buildozer.spec`

Cell này mở hộp thoại chọn file. Anh chọn **cả 2 file** từ thư mục workspace rồi nhấn Open.

In [ ]:
from google.colab import files
import os
for f in ['kivy_app.py', 'buildozer.spec', 'main.py']:
    if os.path.exists(f):
        os.remove(f)
        print('Đã xóa file cũ:', f)
print('\n👉 Chọn 2 file: kivy_app.py + buildozer.spec\n')
uploaded = files.upload()
have = set(os.listdir('.'))
missing = [f for f in ['kivy_app.py', 'buildozer.spec'] if f not in have]
if missing:
    print('\n❌ THIẾU FILE:', missing)
    print('Upload lại nhé.')
else:
    print('\n✅ kivy_app.py:', round(os.path.getsize('kivy_app.py')/1024, 1), 'KB')
    print('✅ buildozer.spec:', round(os.path.getsize('buildozer.spec')/1024, 1), 'KB')
    print('\nSẵn sàng build → chạy Bước 3')

## 🔑 Bước 3: Sửa `buildozer.spec` cho Colab (bỏ pin Python)

Colab chạy Python 3.14.2. Nếu `buildozer.spec` pin `python3==3.11.6` thì sẽ fail với lỗi:
`python3 should have same version as hostpython3, 3.11.6 != 3.14.2`

Cell này tự sửa bằng `sed` rồi in kết quả để anh kiểm tra.

In [ ]:
!sed -i 's/python3==3.11.6/python3/' buildozer.spec
!echo '=== Dòng requirements hiện tại ==='
!grep '^requirements' buildozer.spec
!echo ''
!echo '✅ Nếu thấy "requirements = python3, kivy==2.3.0, cython==0.29.36" → OK, chạy Bước 4'

## 🔨 Bước 4: Build APK

⏱️ **Lần đầu 30-45 phút** (tải SDK + NDK + compile Kivy). Lần sau ~10-15 phút.

**Giải thích lệnh:**
- `yes |` — tự động trả lời "y" cho câu hỏi "Buildozer is running as root! Are you sure?"
- `--accept-eula` — auto accept Android SDK licenses
- **Không có `tail`** để xem full log streaming ra theo thời gian thực
- `tee build.log` — vẫn save log vào file để xem lại sau

In [ ]:
import time
t0 = time.time()
!rm -rf ~/.buildozer  # Xóa cache cũ (build trước dùng Python 3.14 fail)
!yes | buildozer --accept-eula android debug 2>&1 | tee build.log
elapsed = time.time() - t0
mins = int(elapsed // 60)
secs = int(elapsed % 60)
print('\n⏱️ Thời gian build:', mins, 'phút', secs, 'giây')

## 🔍 Bước 5: Xem log nếu build fail

Nếu Bước 4 không tạo APK, copy **50 dòng cuối** của log gửi cho mình debug.

In [ ]:
!tail -50 build.log

## 📥 Bước 6: Tải APK về máy

Cell này tìm file APK trong các đường dẫn có thể rồi mở hộp thoại download.

In [ ]:
import os, glob
from google.colab import files
apk_candidates = glob.glob('bin/*.apk') + glob.glob('.buildozer/android/platform/build-*/dists/*/build/outputs/apk/**/*.apk', recursive=True)
print('Tìm thấy các file APK:')
for apk in apk_candidates:
    size_mb = os.path.getsize(apk) / 1024 / 1024
    print('  📦', apk, ' (', round(size_mb, 2), 'MB)')
if not apk_candidates:
    print('\n❌ KHÔNG TÌM THẤY APK — quay lại Bước 5 xem log')
else:
    debug_apks = [a for a in apk_candidates if 'debug' in a]
    target = sorted(debug_apks, key=lambda a: os.path.getsize(a))[0] if debug_apks else apk_candidates[0]
    size_mb = os.path.getsize(target) / 1024 / 1024
    print('\n🎯 File sẽ tải:', target, '(', round(size_mb, 2), 'MB)')
    print('Đang mở hộp thoại download...\n')
    files.download(target)
    print('\n✅ Đã tải APK thành công!')
    print('👉 Copy file .apk vào điện thoại Android → mở file → cài đặt')